# Bronze validation with Great Expectations (streaming)

Purpose: validate the routed Bronze valid stream incrementally, emit append-only observability outputs, and publish append-only validated/quarantine sinks for downstream streaming.


In [0]:
%pip install great-expectations==0.18.21


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
import os
import traceback
from datetime import datetime, timezone
from functools import reduce

import great_expectations as gx
from great_expectations.data_context import FileDataContext
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "hsl"

SOURCE_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.bronze_vehicle_positions_valid"
BASE_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/bronze"

GE_RESULTS_PATH = f"{BASE_PATH}/bronze_ge/ge_results/hsl_vehicle_positions"
GE_DETAIL_PATH = f"{BASE_PATH}/bronze_ge/ge_details/hsl_vehicle_positions"
FAILED_ROW_SAMPLE_PATH = f"{BASE_PATH}/bronze_ge/failed_row_samples/hsl_vehicle_positions"
FAILED_RULE_METRICS_PATH = f"{BASE_PATH}/bronze_ge/rule_metrics/hsl_vehicle_positions"
GATE_RESULTS_PATH = f"{BASE_PATH}/bronze_ge/gate_results/hsl_vehicle_positions"

VALIDATED_BRONZE_OUTPUT_PATH = f"{BASE_PATH}/bronze_validated_stream/hsl_vehicle_positions"
QUARANTINE_OUTPUT_PATH = f"{BASE_PATH}/bronze_ge_quarantine_stream/hsl_vehicle_positions"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_ge_validate_stream_hsl_vehicle_positions"

BRONZE_VALIDATED_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.bronze_vehicle_positions_validated_stream"
BRONZE_VALIDATION_QUAR_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.bronze_vehicle_positions_ge_quarantine_stream"
BRONZE_GE_RESULTS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.bronze_ge_results_hsl_vehicle_positions"
BRONZE_GE_DETAILS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.bronze_ge_details_hsl_vehicle_positions"
BRONZE_GE_SAMPLES_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.bronze_ge_failed_samples_hsl_vehicle_positions"
BRONZE_RULE_METRICS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.bronze_ge_rule_metrics_hsl_vehicle_positions"
BRONZE_GATE_RESULTS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.bronze_gate_results_hsl_vehicle_positions"

TRIGGER_INTERVAL = "10 seconds"
MAX_FAILED_SAMPLE_ROWS_PER_RULE = 100

MAX_QUARANTINE_RATE = 0.01
MAX_CRITICAL_ROWS = 0
MAX_HIGH_ROWS = 0
MAX_MEDIUM_WARNING_RATE = 0.05

CONTEXT_ROOT_DIR = "/dbfs/great_expectations/bronze_validate_ge_stream_ctx"
DATASOURCE_NAME = "bronze_runtime_spark_stream"
EXPECTATION_SUITE_NAME = "bronze_valid_post_route_stream_suite"
DATA_ASSET_NAME = "bronze_vehicle_positions_valid_stream_asset"
QUERY_NAME = "bronze_vehicle_positions_ge_validate_stream"


def filter_to_target_ingest_date(df):
    if "ingest_date" not in df.columns:
        raise ValueError("Expected ingest_date column before validation date filtering.")
    return df.where(F.to_date(F.col("ingest_date")) == F.current_date())


In [0]:
# tables_and_paths = [
#     (BRONZE_VALIDATED_TABLE, VALIDATED_BRONZE_OUTPUT_PATH),
#     (BRONZE_VALIDATION_QUAR_TABLE, QUARANTINE_OUTPUT_PATH),
#     (BRONZE_GE_RESULTS_TABLE, GE_RESULTS_PATH),
#     (BRONZE_GE_DETAILS_TABLE, GE_DETAIL_PATH),
#     (BRONZE_GE_SAMPLES_TABLE, FAILED_ROW_SAMPLE_PATH),
#     (BRONZE_RULE_METRICS_TABLE, FAILED_RULE_METRICS_PATH),
#     (BRONZE_GATE_RESULTS_TABLE, GATE_RESULTS_PATH),
# ]

# for table, path in tables_and_paths:
#     spark.sql(f"DROP TABLE IF EXISTS {table}")
#     dbutils.fs.rm(path, True)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

for q in spark.streams.active:
    if q.name == QUERY_NAME:
        q.stop()

base_schema = spark.table(SOURCE_TABLE).schema
sink_schema = StructType(list(base_schema.fields) + [
    StructField("failed_rule_ids", ArrayType(StringType()), True),
    StructField("failed_severities", ArrayType(StringType()), True),
    StructField("failed_rule_descriptions", ArrayType(StringType()), True),
    StructField("failed_rule_count", LongType(), True),
    StructField("validation_run_id", StringType(), True),
    StructField("validation_batch_id", LongType(), True),
    StructField("validation_scope", StringType(), True),
    StructField("validation_ts", TimestampType(), True),
    StructField("validation_date", StringType(), True),
    StructField("validation_status", StringType(), True),
])

def precreate_sink(path: str, table_name: str):
    if not DeltaTable.isDeltaTable(spark, path):
        (
            spark.createDataFrame([], sink_schema)
            .write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("ingest_date")
            .save(path)
        )
    spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {table_name}
    USING DELTA
    LOCATION "{path}"
    ''')

precreate_sink(VALIDATED_BRONZE_OUTPUT_PATH, BRONZE_VALIDATED_TABLE)
precreate_sink(QUARANTINE_OUTPUT_PATH, BRONZE_VALIDATION_QUAR_TABLE)


In [0]:
gate_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("validated_table", StringType(), True),
    StructField("run_ts_utc", TimestampType(), True),
    StructField("validation_date", StringType(), True),
    StructField("validation_scope", StringType(), True),
    StructField("input_row_count", LongType(), True),
    StructField("validated_row_count", LongType(), True),
    StructField("quarantined_row_count", LongType(), True),
    StructField("quarantine_rate", DoubleType(), True),
    StructField("critical_failed_rows", LongType(), True),
    StructField("high_failed_rows", LongType(), True),
    StructField("medium_failed_rows", LongType(), True),
    StructField("low_failed_rows", LongType(), True),
    StructField("warning_rate", DoubleType(), True),
    StructField("gate_status", StringType(), True),
    StructField("gate_reason", StringType(), True),
    StructField("silver_allowed", BooleanType(), True),
    StructField("validated_output_path", StringType(), True),
    StructField("quarantine_output_path", StringType(), True),
])


def register_table(path: str, table_name: str):
    spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {table_name}
    USING DELTA
    LOCATION "{path}"
    ''')


def write_append(df, path: str, table_name: str, merge_schema: bool = True, partition_cols: list[str] | None = None):
    if not DeltaTable.isDeltaTable(spark, path):
        writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if merge_schema:
            writer = writer.option("mergeSchema", "true")
        if partition_cols:
            writer = writer.partitionBy(*partition_cols)
        writer.save(path)
    else:
        writer = df.write.format("delta").mode("append")
        if merge_schema:
            writer = writer.option("mergeSchema", "true")
        if partition_cols:
            writer = writer.partitionBy(*partition_cols)
        writer.save(path)
    register_table(path, table_name)


def write_gate_results(rows):
    write_append(spark.createDataFrame(rows, schema=gate_schema), GATE_RESULTS_PATH, BRONZE_GATE_RESULTS_TABLE, merge_schema=False)


def union_by_name_allow_missing(dfs):
    if not dfs:
        return None
    return reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), dfs)


def gx_to_dict(obj):
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "to_json_dict"):
        return obj.to_json_dict()
    if hasattr(obj, "to_dict"):
        return obj.to_dict()
    return {}

def get_gx_context():
    os.makedirs(CONTEXT_ROOT_DIR, exist_ok=True)
    print(json.dumps({
        "layer": "bronze_validate_ge",
        "message": "Initializing Great Expectations context",
        "context_root_dir": CONTEXT_ROOT_DIR,
    }, default=str))
    context = FileDataContext.create(project_root_dir=CONTEXT_ROOT_DIR)
    return context


def apply_bronze_expectations(validator):
    validator.expect_column_to_exist("topic")
    validator.expect_column_to_exist("partition")
    validator.expect_column_to_exist("offset")
    validator.expect_column_to_exist("eventhub_enqueued_ts")
    validator.expect_column_to_exist("raw_json")
    validator.expect_column_to_exist("bronze_ingest_ts")
    validator.expect_column_to_exist("ingest_date")
    validator.expect_column_to_exist("parse_ok")
    validator.expect_column_to_exist("parse_error")
    validator.expect_column_to_exist("source")
    validator.expect_column_to_exist("producer_ingest_ts_utc")
    validator.expect_column_to_exist("mqtt_topic")
    validator.expect_column_to_exist("event_type")
    validator.expect_column_to_exist("transport_mode")

    validator.expect_column_values_to_not_be_null("topic")
    validator.expect_column_values_to_not_be_null("partition")
    validator.expect_column_values_to_not_be_null("offset")
    validator.expect_column_values_to_not_be_null("eventhub_enqueued_ts")
    validator.expect_column_values_to_not_be_null("raw_json")
    validator.expect_column_values_to_not_be_null("bronze_ingest_ts")
    validator.expect_column_values_to_not_be_null("ingest_date")
    validator.expect_column_values_to_be_in_set("parse_ok", [True], mostly=1.0)
    validator.expect_column_values_to_be_null("parse_error")
    validator.expect_column_values_to_be_in_set("source", ["hsl_hfp_mqtt"])
    validator.expect_column_values_to_be_in_set("event_type", ["vp"])
    validator.expect_column_values_to_not_be_null("mqtt_topic")
    validator.expect_column_values_to_not_be_null("transport_mode")
    # producer_ingest_ts_utc should be in ISO-8601 UTC format
    validator.expect_column_values_to_match_regex(
        "producer_ingest_ts_utc",
        r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?Z$",
    )
    validator.expect_compound_columns_to_be_unique(["partition", "offset"])


def run_bronze_ge_validation(batch_df, batch_run_id: str, batch_run_ts):
    context = get_gx_context()
    context.add_or_update_expectation_suite(expectation_suite_name=EXPECTATION_SUITE_NAME)
    datasource = context.sources.add_or_update_spark(name=DATASOURCE_NAME)
    asset = datasource.add_dataframe_asset(name=f"{DATA_ASSET_NAME}_{batch_run_id}")
    batch_request = asset.build_batch_request(dataframe=batch_df)
    validator = context.get_validator(batch_request=batch_request, expectation_suite_name=EXPECTATION_SUITE_NAME)
    apply_bronze_expectations(validator)
    validator.save_expectation_suite(discard_failed_expectations=False)
    validation_result = validator.validate(
        run_id={"run_name": batch_run_id, "run_time": batch_run_ts.isoformat()},
        data_context=context,
    )
    return gx_to_dict(validation_result)


In [0]:
rule_specs = [
    {"rule_id": "critical_topic_null", "severity": "critical", "description": "Kafka/Event Hubs topic must not be null.", "fail_condition": F.col("topic").isNull(), "quarantine_row": True},
    {"rule_id": "critical_partition_null", "severity": "critical", "description": "Kafka partition must not be null.", "fail_condition": F.col("partition").isNull(), "quarantine_row": True},
    {"rule_id": "critical_offset_null", "severity": "critical", "description": "Kafka offset must not be null.", "fail_condition": F.col("offset").isNull(), "quarantine_row": True},
    {"rule_id": "critical_eventhub_enqueued_ts_null", "severity": "critical", "description": "Event Hubs enqueue timestamp must not be null.", "fail_condition": F.col("eventhub_enqueued_ts").isNull(), "quarantine_row": True},
    {"rule_id": "critical_raw_json_null", "severity": "critical", "description": "Raw JSON payload must not be null.", "fail_condition": F.col("raw_json").isNull(), "quarantine_row": True},
    {"rule_id": "critical_bronze_ingest_ts_null", "severity": "critical", "description": "Bronze ingest timestamp must not be null.", "fail_condition": F.col("bronze_ingest_ts").isNull(), "quarantine_row": True},
    {"rule_id": "critical_parse_ok_false", "severity": "critical", "description": "Rows in the valid Bronze table should still have parse_ok = true.", "fail_condition": F.coalesce(F.col("parse_ok"), F.lit(False)) == F.lit(False), "quarantine_row": True},
    {"rule_id": "high_parse_error_present", "severity": "high", "description": "Rows in the valid Bronze table should not carry parse_error.", "fail_condition": F.col("parse_error").isNotNull(), "quarantine_row": True},
    {"rule_id": "high_source_not_hsl_hfp_mqtt", "severity": "high", "description": "Source should be hsl_hfp_mqtt.", "fail_condition": F.coalesce(F.col("source"), F.lit("")) != F.lit("hsl_hfp_mqtt"), "quarantine_row": True},
    {"rule_id": "high_event_type_not_vp", "severity": "high", "description": "Event type should be vp for vehicle positions.", "fail_condition": F.coalesce(F.col("event_type"), F.lit("")) != F.lit("vp"), "quarantine_row": True},
    {"rule_id": "high_mqtt_topic_null", "severity": "high", "description": "MQTT topic should be present for traceability.", "fail_condition": F.col("mqtt_topic").isNull(), "quarantine_row": True},
    {"rule_id": "high_transport_mode_null", "severity": "high", "description": "Transport mode is missing.", "fail_condition": F.col("transport_mode").isNull(), "quarantine_row": True},
    {"rule_id": "medium_producer_ts_bad_format", "severity": "medium", "description": "Producer ingest timestamp should be ISO-8601 UTC ending with Z.", "fail_condition": F.col("producer_ingest_ts_utc").isNotNull() & (~F.col("producer_ingest_ts_utc").rlike(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?Z$")), "quarantine_row": False},
]


In [0]:
def write_bronze_validation_batch(batch_df, batch_id: int):
    if batch_df.isEmpty():
        return

    batch_df = filter_to_target_ingest_date(batch_df)
    if batch_df.isEmpty():
        return

    batch_run_ts = datetime.now(timezone.utc)
    batch_run_id = f"bronze_ge_stream_{batch_id}_{batch_run_ts.strftime('%Y%m%d_%H%M%S')}"
    validation_date = batch_run_ts.date().isoformat()
    validation_scope = f"microbatch_id={batch_id}"

    working_df = batch_df.cache()

    try:
        row_count = working_df.count()
        print(json.dumps({
            "layer": "bronze_validate_ge",
            "message": "Starting foreachBatch validation",
            "batch_id": int(batch_id),
            "batch_run_id": batch_run_id,
            "row_count": int(row_count),
            "validation_scope": validation_scope,
            "context_root_dir": CONTEXT_ROOT_DIR,
        }, default=str))

        validation_result = run_bronze_ge_validation(working_df, batch_run_id, batch_run_ts)
        print(json.dumps({
            "layer": "bronze_validate_ge",
            "batch_run_id": batch_run_id,
        }, default=str))
        stats = validation_result.get("statistics", {}) or {}

        # one summary row per batch, including overall evaluated/ successful and unsuccessful expectations and success percent
        summary_row = [{
            "run_id": batch_run_id,
            "validated_table": SOURCE_TABLE,
            "validation_scope": validation_scope,
            "validation_date": validation_date,
            "run_ts_utc": batch_run_ts,
            "row_count": int(row_count),
            "success": bool(validation_result.get("success", False)),
            "evaluated_expectations": int(stats.get("evaluated_expectations", 0) or 0),
            "successful_expectations": int(stats.get("successful_expectations", 0) or 0),
            "unsuccessful_expectations": int(stats.get("unsuccessful_expectations", 0) or 0),
            "success_percent": float(stats.get("success_percent", 0.0) or 0.0),
        }]
        write_append(spark.createDataFrame(summary_row), GE_RESULTS_PATH, BRONZE_GE_RESULTS_TABLE)

        # expectation-level details for each run
        detail_rows = []
        for result in validation_result.get("results", []):
            cfg = result.get("expectation_config", {}) or {}
            payload = result.get("result", {}) or {}
            detail_rows.append({
                "run_id": batch_run_id,
                "validated_table": SOURCE_TABLE,
                "validation_date": validation_date,
                "run_ts_utc": batch_run_ts,
                "expectation_type": cfg.get("expectation_type"),
                "kwargs_json": json.dumps((cfg.get("kwargs", {}) or {}), default=str),
                "success": bool(result.get("success", False)),
                "unexpected_count": int(payload.get("unexpected_count", 0) or 0),
                "unexpected_percent": float(payload.get("unexpected_percent", 0.0) or 0.0),
                "element_count": int(payload.get("element_count", row_count) or row_count),
            })
        if detail_rows:
            write_append(spark.createDataFrame(detail_rows), GE_DETAIL_PATH, BRONZE_GE_DETAILS_TABLE)

        flagged_df = working_df
        for spec in rule_specs:
            flagged_df = flagged_df.withColumn(
                f"_fail_{spec['rule_id']}",
                F.when(spec["fail_condition"], F.lit(1)).otherwise(F.lit(0)),
            )
        counts_row = flagged_df.agg(*[
            F.sum(F.col(f"_fail_{spec['rule_id']}")).alias(spec["rule_id"])
            for spec in rule_specs
        ]).collect()[0]

        
        failed_dfs = []
        # per-rule failure counts from Spark rule extraction
        rule_metric_rows = []
        for spec in rule_specs:
            fail_count = int(counts_row[spec["rule_id"]] or 0)
            rule_metric_rows.append({
                "run_id": batch_run_id,
                "validation_date": validation_date,
                "validation_scope": validation_scope,
                "run_ts_utc": batch_run_ts,
                "rule_id": spec["rule_id"],
                "severity": spec["severity"],
                "quarantine_row": bool(spec["quarantine_row"]),
                "failed_row_count": fail_count,
                "failed_row_rate": float(fail_count) / float(row_count) if row_count else 0.0,
                "rule_description": spec["description"],
            })
            if fail_count > 0:
                failed_dfs.append(
                    working_df
                    .filter(spec["fail_condition"])
                    .withColumn("rule_id", F.lit(spec["rule_id"]))
                    .withColumn("severity", F.lit(spec["severity"]))
                    .withColumn("rule_description", F.lit(spec["description"]))
                    .withColumn("quarantine_row", F.lit(spec["quarantine_row"]))
                    .withColumn("validation_run_id", F.lit(batch_run_id))
                    .withColumn("validation_batch_id", F.lit(int(batch_id)).cast("bigint"))
                    .withColumn("validation_scope", F.lit(validation_scope))
                    .withColumn("validation_ts", F.lit(batch_run_ts))
                    .withColumn("validation_date", F.lit(validation_date))
                )

        duplicate_keys_df = (
            working_df.groupBy("topic", "partition", "offset")
            .count()
            .filter(F.col("count") > 1)
            .select("topic", "partition", "offset")
        )
        if duplicate_keys_df.count() > 0:
            duplicate_fail_df = (
                working_df.alias("b")
                .join(duplicate_keys_df.alias("d"), on=["topic", "partition", "offset"], how="inner")
                .withColumn("rule_id", F.lit("critical_duplicate_partition_offset"))
                .withColumn("severity", F.lit("critical"))
                .withColumn("rule_description", F.lit("(topic, partition, offset) must be unique within the Bronze validation microbatch."))
                .withColumn("quarantine_row", F.lit(True))
                .withColumn("validation_run_id", F.lit(batch_run_id))
                .withColumn("validation_batch_id", F.lit(int(batch_id)).cast("bigint"))
                .withColumn("validation_scope", F.lit(validation_scope))
                .withColumn("validation_ts", F.lit(batch_run_ts))
                .withColumn("validation_date", F.lit(validation_date))
            )
            duplicate_row_count = duplicate_fail_df.count()
            failed_dfs.append(duplicate_fail_df)
            rule_metric_rows.append({
                "run_id": batch_run_id,
                "validation_date": validation_date,
                "validation_scope": validation_scope,
                "run_ts_utc": batch_run_ts,
                "rule_id": "critical_duplicate_partition_offset",
                "severity": "critical",
                "quarantine_row": True,
                "failed_row_count": int(duplicate_row_count),
                "failed_row_rate": float(duplicate_row_count) / float(row_count) if row_count else 0.0,
                "rule_description": "(topic, partition, offset) must be unique within the Bronze validation microbatch.",
            })

        write_append(spark.createDataFrame(rule_metric_rows), FAILED_RULE_METRICS_PATH, BRONZE_RULE_METRICS_TABLE)

        all_failed_rows_df = union_by_name_allow_missing(failed_dfs)
        if all_failed_rows_df is None:
            all_failed_rows_df = spark.createDataFrame(
                [],
                schema=(
                    working_df
                    .withColumn("rule_id", F.lit(None).cast("string"))
                    .withColumn("severity", F.lit(None).cast("string"))
                    .withColumn("rule_description", F.lit(None).cast("string"))
                    .withColumn("quarantine_row", F.lit(None).cast("boolean"))
                    .withColumn("validation_run_id", F.lit(None).cast("string"))
                    .withColumn("validation_batch_id", F.lit(None).cast("bigint"))
                    .withColumn("validation_scope", F.lit(None).cast("string"))
                    .withColumn("validation_ts", F.lit(None).cast("timestamp"))
                    .withColumn("validation_date", F.lit(None).cast("string"))
                    .schema
                ),
            )
        # sampled failed rows for debugging and illustration
        failed_samples_df = (
            all_failed_rows_df
            .filter(F.col("rule_id").isNotNull())
            .withColumn(
                "sample_rank",
                F.row_number().over(
                    Window.partitionBy("rule_id").orderBy(
                        F.col("bronze_ingest_ts").desc_nulls_last(),
                        F.col("offset").desc_nulls_last(),
                    )
                ),
            )
            .filter(F.col("sample_rank") <= F.lit(MAX_FAILED_SAMPLE_ROWS_PER_RULE))
            .drop("sample_rank")
        )
        if not failed_samples_df.isEmpty():
            write_append(failed_samples_df, FAILED_ROW_SAMPLE_PATH, BRONZE_GE_SAMPLES_TABLE)

        failed_rollup_df = (
            all_failed_rows_df
            .filter(F.col("rule_id").isNotNull())
            .groupBy("topic", "partition", "offset")
            .agg(
                F.collect_set("rule_id").alias("failed_rule_ids"),
                F.collect_set("severity").alias("failed_severities"),
                F.collect_set("rule_description").alias("failed_rule_descriptions"),
                F.max(F.when(F.col("quarantine_row") == True, F.lit(1)).otherwise(F.lit(0))).alias("should_quarantine"),
            )
        )

        empty_array = F.expr("array()").cast("array<string>")
        classified_df = (
            working_df
            .join(failed_rollup_df, on=["topic", "partition", "offset"], how="left")
            .withColumn("failed_rule_ids", F.when(F.col("failed_rule_ids").isNull(), empty_array).otherwise(F.col("failed_rule_ids")))
            .withColumn("failed_severities", F.when(F.col("failed_severities").isNull(), empty_array).otherwise(F.col("failed_severities")))
            .withColumn("failed_rule_descriptions", F.when(F.col("failed_rule_descriptions").isNull(), empty_array).otherwise(F.col("failed_rule_descriptions")))
            .withColumn("failed_rule_count", F.size(F.col("failed_rule_ids")).cast("bigint"))
            .withColumn("validation_run_id", F.lit(batch_run_id))
            .withColumn("validation_batch_id", F.lit(int(batch_id)).cast("bigint"))
            .withColumn("validation_scope", F.lit(validation_scope))
            .withColumn("validation_ts", F.lit(batch_run_ts))
            .withColumn("validation_date", F.lit(validation_date))
            .withColumn(
                "validation_status",
                F.when(F.coalesce(F.col("should_quarantine"), F.lit(0)) == F.lit(1), F.lit("quarantined"))
                .otherwise(F.lit("validated"))
            )
            .drop("should_quarantine")
        )


        validated_df = classified_df.filter(F.col("validation_status") == "validated").cache()
        quarantine_df = classified_df.filter(F.col("validation_status") == "quarantined").cache()
        try:
            validated_row_count = validated_df.count()
            quarantined_row_count = quarantine_df.count()
            quarantine_rate = float(quarantined_row_count) / float(row_count) if row_count else 0.0
            write_append(validated_df, VALIDATED_BRONZE_OUTPUT_PATH, BRONZE_VALIDATED_TABLE, merge_schema=False, partition_cols=["ingest_date"])
            write_append(quarantine_df, QUARANTINE_OUTPUT_PATH, BRONZE_VALIDATION_QUAR_TABLE, merge_schema=False, partition_cols=["ingest_date"])
        finally:
            validated_df.unpersist()
            quarantine_df.unpersist()

        severity_rollup = {
            row["severity"]: row["affected_rows"]
            for row in (
                all_failed_rows_df
                .filter(F.col("rule_id").isNotNull())
                .select("severity", "topic", "partition", "offset")
                .distinct()
                .groupBy("severity")
                .agg(F.count("*").alias("affected_rows"))
                .collect()
            )
        }

        critical_failed_rows = int(severity_rollup.get("critical", 0))
        high_failed_rows = int(severity_rollup.get("high", 0))
        medium_failed_rows = int(severity_rollup.get("medium", 0))
        low_failed_rows = int(severity_rollup.get("low", 0))
        warning_rows = medium_failed_rows + low_failed_rows
        warning_rate = float(warning_rows) / float(row_count) if row_count else 0.0

        block_reasons = []
        warn_reasons = []
        if critical_failed_rows > MAX_CRITICAL_ROWS:
            block_reasons.append(f"critical_failed_rows={critical_failed_rows} exceeds threshold {MAX_CRITICAL_ROWS}")
        if high_failed_rows > MAX_HIGH_ROWS:
            block_reasons.append(f"high_failed_rows={high_failed_rows} exceeds threshold {MAX_HIGH_ROWS}")
        if quarantine_rate > MAX_QUARANTINE_RATE:
            block_reasons.append(f"quarantine_rate={quarantine_rate:.6f} exceeds threshold {MAX_QUARANTINE_RATE:.6f}")
        if medium_failed_rows > 0:
            warn_reasons.append(f"medium_failed_rows={medium_failed_rows}")
        if low_failed_rows > 0:
            warn_reasons.append(f"low_failed_rows={low_failed_rows}")
        if warning_rate > MAX_MEDIUM_WARNING_RATE:
            warn_reasons.append(f"warning_rate={warning_rate:.6f} exceeds threshold {MAX_MEDIUM_WARNING_RATE:.6f}")

        if block_reasons:
            gate_status = "BLOCK"
            silver_allowed = False
            gate_reason = "; ".join(block_reasons)
        elif warn_reasons:
            gate_status = "WARN"
            silver_allowed = True
            gate_reason = "; ".join(warn_reasons)
        else:
            gate_status = "PASS"
            silver_allowed = True
            gate_reason = "No failed rows detected above warning threshold."

        gate_rows = [{
            "run_id": batch_run_id,
            "validated_table": SOURCE_TABLE,
            "run_ts_utc": batch_run_ts,
            "validation_date": validation_date,
            "validation_scope": validation_scope,
            "input_row_count": int(row_count),
            "validated_row_count": int(validated_row_count),
            "quarantined_row_count": int(quarantined_row_count),
            "quarantine_rate": float(quarantine_rate),
            "critical_failed_rows": int(critical_failed_rows),
            "high_failed_rows": int(high_failed_rows),
            "medium_failed_rows": int(medium_failed_rows),
            "low_failed_rows": int(low_failed_rows),
            "warning_rate": float(warning_rate),
            "gate_status": gate_status,
            "gate_reason": gate_reason,
            "silver_allowed": bool(silver_allowed),
            "validated_output_path": VALIDATED_BRONZE_OUTPUT_PATH,
            "quarantine_output_path": QUARANTINE_OUTPUT_PATH,
        }]
        write_gate_results(gate_rows)

        try:
            dbutils.jobs.taskValues.set(key="bronze_gate_status", value=gate_status)
            dbutils.jobs.taskValues.set(key="bronze_gate_reason", value=gate_reason[:1000])
            dbutils.jobs.taskValues.set(key="bronze_validated_path", value=VALIDATED_BRONZE_OUTPUT_PATH)
            dbutils.jobs.taskValues.set(key="bronze_quarantine_path", value=QUARANTINE_OUTPUT_PATH)
            dbutils.jobs.taskValues.set(key="bronze_validation_run_id", value=batch_run_id)
        except Exception:
            pass
    except Exception as exc:
        print(json.dumps({
            "layer": "bronze_validate_ge",
            "message": "foreachBatch failed",
            "batch_id": int(batch_id),
            "batch_run_id": batch_run_id,
            "validation_scope": validation_scope,
            "context_root_dir": CONTEXT_ROOT_DIR,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "traceback": traceback.format_exc(),
        }, default=str))
        raise
    finally:
        working_df.unpersist()


In [0]:
bronze_validation_query = (
    spark.readStream.table(SOURCE_TABLE)
    .writeStream
    .queryName(QUERY_NAME)
    .foreachBatch(write_bronze_validation_batch)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start()
)


In [0]:
for q in spark.streams.active:
    if q.name == QUERY_NAME:
        print("NAME:", q.name)
        print("ID:", q.id)
        print("IS ACTIVE:", q.isActive)
        print("STATUS:", q.status)
        print("LAST PROGRESS:", q.lastProgress)
        print("EXCEPTION:", q.exception())
        break

bronze_validation_query.awaitTermination()


NAME: bronze_vehicle_positions_ge_validate_stream
ID: c7056b7e-bc1a-434f-b5bf-b9a8d0409812
IS ACTIVE: True
STATUS: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
LAST PROGRESS: None
EXCEPTION: None
{"layer": "bronze_validate_ge", "message": "Starting foreachBatch validation", "batch_id": 17, "batch_run_id": "bronze_ge_stream_17_20260422_050650", "row_count": 9998, "validation_scope": "microbatch_id=17", "context_root_dir": "/dbfs/great_expectations/bronze_validate_ge_stream_ctx"}
{"layer": "bronze_validate_ge", "message": "Initializing Great Expectations context", "context_root_dir": "/dbfs/great_expectations/bronze_validate_ge_stream_ctx"}


/local_disk0/.ephemeral_nfs/envs/pythonEnv-ee9ec771-8792-46b4-9b18-c4eaf39c2473/lib/python3.12/site-packages/great_expectations/data_context/data_context/serializable_data_context.py:225: UserWarning: Warning. An existing `great_expectations.yml` was found here: /dbfs/great_expectations/bronze_validate_ge_stream_ctx/gx.
    - No action was taken.
  warnings.warn(message)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-ee9ec771-8792-46b4-9b18-c4eaf39c2473/lib/python3.12/site-packages/great_expectations/data_context/data_context/serializable_data_context.py:233: UserWarning: Warning. An existing `config_variables.yml` was found here: /dbfs/great_expectations/bronze_validate_ge_stream_ctx/gx/uncommitted.
    - No action was taken.
  warnings.warn(message)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-ee9ec771-8792-46b4-9b18-c4eaf39c2473/lib/python3.12/site-packages/great_expectations/expectations/expectation.py:1519: UserWarning: `result_format` configured at the Validator-level will not be persi

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/76 [00:00<?, ?it/s]

{"layer": "bronze_validate_ge", "batch_run_id": "bronze_ge_stream_23_20260422_054739"}


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can